In [0]:
%sql

select sigList.signal_name, siglist.signal_path_in_source_system, sigList.description, sigList.power_plant_name, sigList.equipment_name, sigList.metering_point_object, sigList.scada_tag from espark_industrial_assets_and_processes_prod_exposition.`eu-central-1_rendataops_solar-operations-dat_domain`.signals_with_context as sigList where siglist.power_plant_name IN ("Hill 1" ) and sigList.signal_name like "%planeofarray_irradiance%"

In [0]:
%sql
select distinct sigList.signal_name, siglist.signal_path_in_source_system, sigList.description, sigList.equipment_name, sigList.power_plant_name from espark_industrial_assets_and_processes_prod_exposition.`eu-central-1_rendataops_solar-operations-dat_domain`.signals_with_context as sigList where siglist.power_plant_name IN ("Hill 1" )  and sigList.signal_name IN ("planeofarray_irradiance", "rear_plane_of_array_irradiance", "global_horizontal_irradiance")

In [0]:
%pip install pvlib==0.16.1 plotly
%restart_python

In [0]:
%sql
--* Base readings for the signal, currently uses "Signal" widget at the top of the notebook but will extend to all later
with base as (
  select main.equipment_id, main.equipment_name, main.equipment_type, main.plant_id, main.power_plant_name, main.value, main.timestamp, main.timezone, main.id, main.scada_tag, main.signal_name
  from espark_industrial_assets_and_processes_prod_exposition.`eu-central-1_rendataops_ren-ops-analytics_domain`.solar_measurements_standardised_with_context as main
  where main.power_plant_name IN ("Hill 1")
    and main.timestamp between '2026-09-01T00:00:00.000+00:00' and '2026-11-01T18:18:19.000+00:00'
    and main.signal_name = :signal_name
),

--* Frozen Check on all readings, per equipment and signal

-- Consecutive timestamps with an identical value form one run
run_start as (
  select b.*,
         CASE WHEN lag(b.value) over (partition by b.equipment_id, b.signal_name order by b.timestamp) <=> b.value
              THEN 0 ELSE 1 END as is_new_run
  from base b
),
-- Group same values
run_id as (
  select r.*,
         sum(r.is_new_run) over (partition by r.equipment_id, r.signal_name order by r.timestamp
                                 rows between unbounded preceding and current row) as frozen_run_id
  from run_start r
),
-- Split that group into indexes
run_stats as (
  select r.*,
         count(*)     over (partition by r.equipment_id, r.signal_name, r.frozen_run_id) as frozen_run_length,
         row_number() over (partition by r.equipment_id, r.signal_name, r.frozen_run_id order by r.timestamp) as position_in_run
  from run_id r
),
-- Flag groups that are greater than 10 but dont flag the first value
with_frozen as (
  select s.* except (is_new_run),
         CASE WHEN s.frozen_run_length > 10 AND s.position_in_run > 1 THEN 1 ELSE 0 END as is_frozen
  from run_stats s
),

-- Bounds Check (0-1500 W/m2)
with_bounds as (
  select f.*,
         -- 1 = within bounds, 0 = out of bounds
         CASE WHEN f.value >= 0 and f.value <= 1500 THEN 1 ELSE 0 END as within_bounds
  from with_frozen f
),

--* Peer Median Check, only among readings that passed frozen and bounds checks
-- Median of all equipment's passing readings for the same signal in the next 10 minutes [t, t + 10 min)
passed as (
  select * from with_bounds where is_frozen = 0 and within_bounds = 1
),
peer as (
  select b.id,
         median(o.value)                as peer_median_next_10min,
         count(o.value)                 as peer_reading_count,
         count(distinct o.equipment_id) as peer_equipment_count
  from passed b
  join passed o
    on  o.plant_id    = b.plant_id
    and o.signal_name = b.signal_name
    and o.timestamp  >= b.timestamp
    and o.timestamp  <  b.timestamp + INTERVAL 10 MINUTES
  group by b.id
),
       
-- 1 = within +/-10% of peer median, 0 = outside, NULL = failed frozen/bounds check (not evaluated)
peer_check as (select f.*,
       p.peer_median_next_10min,
       p.peer_reading_count,
       p.peer_equipment_count,
       CASE WHEN p.peer_median_next_10min IS NULL THEN NULL
            WHEN abs(f.value - p.peer_median_next_10min) <= 0.10 * abs(p.peer_median_next_10min) THEN 1
            ELSE 0 END as within_peer_median_10pct
from with_bounds f
left join peer p on p.id = f.id
order by f.timestamp, f.equipment_name)

--* Per timestamp: if more than 50% of the peer-evaluated readings are within +/-10% of the peer median,
--* average only those readings; otherwise average all readings that passed the frozen and bounds checks.
-- avg() ignores NULLs, so the share is over readings that were peer-evaluated (passed frozen and bounds)
select pc.*,
       avg(pc.within_peer_median_10pct) over (partition by pc.timestamp, pc.signal_name) as share_within_peer_median,
       CASE WHEN avg(pc.within_peer_median_10pct) over (partition by pc.timestamp, pc.signal_name) > 0.5
            THEN avg(CASE WHEN pc.is_frozen = 0 AND pc.within_bounds = 1 AND pc.within_peer_median_10pct = 1 THEN pc.value END) over (partition by pc.timestamp, pc.signal_name)
            ELSE avg(CASE WHEN pc.is_frozen = 0 AND pc.within_bounds = 1 THEN pc.value END) over (partition by pc.timestamp, pc.signal_name) END as avg_value
from peer_check as pc



In [0]:
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, LongType,
                               DoubleType, StringType)

# Step 5: interpolate data that is missing for less than 1 hour
# Uses the result of Cell 8 (_sqldf), so run Cell 8 right before this cell.
# Output: the in-memory DataFrame site_minute (not saved to a table), used by the plot cell below.
MAX_MISSING_MINUTES = 60   # gaps with fewer missing minutes than this are linearly interpolated
keys = ["plant_id", "power_plant_name", "signal_name", "timezone"]

# 5a. One value per minute, taken directly from avg_value (Cell 8):
#     measured          = more than 50% of peers within +/-10% of the peer median, mean of those readings
#     averaged_fallback = 50% or less within, mean of all readings that passed frozen and bounds checks
site = (_sqldf
        .withColumn("timestamp", F.date_trunc("minute", "timestamp"))
        .groupBy(*keys, "timestamp")
        .agg(F.avg("avg_value").alias("avg_value"),
             F.avg("share_within_peer_median").alias("share_within_peer_median"))
        .withColumn("clean_value", F.col("avg_value").cast("double"))
        .withColumn("source", F.when(F.col("avg_value").isNull(), F.lit(None))
                               .when(F.col("share_within_peer_median") > 0.5, "measured")
                               .otherwise("averaged_fallback"))
        # pass time to pandas as epoch seconds (UTC), so the session timezone / DST can never shift the minute grid
        .withColumn("ts_epoch", F.unix_timestamp("timestamp"))
        .select(*keys, "ts_epoch", "clean_value", "source"))

# 5b. Per series: build the 1-minute grid, find each gap's length, interpolate only short gaps
out_schema = StructType(
    [f for f in site.schema.fields if f.name in keys] + [
        StructField("ts_epoch", LongType()),
        StructField("final_value", DoubleType()),
        StructField("source", StringType()),
        StructField("gap_missing_minutes", DoubleType()),
    ])

def fill_series(pdf: pd.DataFrame) -> pd.DataFrame:
    g = (pdf.assign(timestamp=pd.to_datetime(pdf["ts_epoch"], unit="s"))   # naive UTC, no DST
            .drop(columns="ts_epoch")
            .set_index("timestamp").sort_index()
            .asfreq("1min"))                                     # complete minute grid
    s = g["clean_value"]

    gap_id  = s.notna().cumsum()                                  # each gap shares an id with the good value before it
    gap_len = s.isna().groupby(gap_id).transform("sum")           # length of the gap each minute belongs to
    filled  = s.interpolate(method="linear", limit_area="inside") # never extrapolates at the start/end

    # interpolate only if the gap is short AND has a good value on both sides
    interp = s.isna() & filled.notna() & (gap_len < MAX_MISSING_MINUTES)

    out = pd.DataFrame({
        "final_value": s.where(~interp, filled),
        "source": g["source"].where(s.notna(), np.where(interp, "interpolated", "missing")),
        "gap_missing_minutes": gap_len.where(s.isna()).astype("float64"),
    }, index=g.index).rename_axis("timestamp").reset_index()

    out["ts_epoch"] = out["timestamp"].astype("datetime64[ns]").astype("int64") // 10**9
    for k in keys:                                                # restore key columns
        out[k] = pdf[k].iloc[0]
    return out[[f.name for f in out_schema.fields]]

site_minute = (site.groupBy(*keys).applyInPandas(fill_series, out_schema)
                   .withColumn("timestamp", F.timestamp_seconds("ts_epoch"))
                   .select(*keys, "timestamp", "final_value", "source", "gap_missing_minutes"))

# Minutes per source (missing = gap of 1 hour or more, to be filled from satellite later)
display(site_minute.groupBy("source").agg(F.count("*").alias("minutes"),
                                          F.max("gap_missing_minutes").alias("longest_gap_minutes"))
                   .orderBy("source"))

In [0]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pyspark.sql import functions as F

signal = dbutils.widgets.get("signal_name")   # same "Signal" widget used by Cell 8

# Uses the result of Cell 8 (_sqldf holds the last SQL cell's result) and site_minute from Step 5,
# so run Cell 8 and then Step 5 right before this cell.
checks = (_sqldf
          .withColumn("local_time", F.from_utc_timestamp("timestamp", F.col("timezone")))
          .select("equipment_name", "local_time", "value", "is_frozen", "within_bounds", "within_peer_median_10pct",
                  "share_within_peer_median", "avg_value")
          .toPandas())

# Cumulative cleaning stages, in the same order as Cell 8
keep_raw    = checks["value"].notna()
keep_frozen = keep_raw    & (checks["is_frozen"] == 0)
keep_bounds = keep_frozen & (checks["within_bounds"] == 1)
keep_median = keep_bounds & (checks["within_peer_median_10pct"] == 1)

# 4. avg_value from Cell 8: one value per timestamp (repeated on every row of that timestamp, so keep one)
#    measured          = more than 50% of peers within +/-10% of the peer median, mean of those readings
#    averaged_fallback = 50% or less within, mean of all readings that passed frozen and bounds checks
avg_pd = (checks[checks["avg_value"].notna()]
          .drop_duplicates("local_time")[["local_time", "avg_value", "share_within_peer_median"]]
          .sort_values("local_time"))
avg_pd["source"] = np.where(avg_pd["share_within_peer_median"] > 0.5, "measured", "averaged_fallback")
avg_title = "4. avg_value per timestamp (>50% within peer median vs fallback)"

# 5. final_value from Step 5: 1-minute series, gaps under 1 hour interpolated
final_pd = (site_minute
            .withColumn("local_time", F.from_utc_timestamp("timestamp", F.col("timezone")))
            .select("local_time", "final_value", "source")
            .toPandas()
            .sort_values("local_time"))
final_title = "5. final_value per minute (Step 5, gaps under 1 h interpolated)"

source_colors = {"measured": "#2ca02c", "averaged_fallback": "#ff7f0e",
                 "interpolated": "#9467bd", "missing": "#d62728"}

stages = [
    ("0. Raw readings",                     keep_raw,    None),
    ("1. After frozen check",               keep_frozen, keep_raw),
    ("2. After 0-1500 W/m² bounds check",   keep_bounds, keep_frozen),
    ("3. After +/-10% peer median check",   keep_median, keep_bounds),
]

# Counts per stage (overall and per weather station)
total = int(keep_raw.sum())
print(f"{'Stage':40s} {'good':>7s} {'removed':>8s} {'% of raw':>9s}")
for title, keep, prev in stages:
    good = int(keep.sum())
    removed = 0 if prev is None else int((prev & ~keep).sum())
    print(f"{title:40s} {good:7d} {removed:8d} {100 * good / total:8.1f}%")
print(f"{'4. avg_value per timestamp':40s} {len(avg_pd):7d} timestamps (of {checks['local_time'].nunique()})")
for src, n in avg_pd["source"].value_counts().items():
    print(f"{'     ' + src:40s} {n:7d}")
print(f"{'5. final_value per minute':40s} {int(final_pd['final_value'].notna().sum()):7d} minutes (of {len(final_pd)})")
for src, n in final_pd["source"].value_counts().items():
    print(f"{'     ' + str(src):40s} {n:7d}")
print()
per_station = {title: checks[keep].groupby("equipment_name").size() for title, keep, _ in stages}
display(pd.DataFrame(per_station).fillna(0).astype(int).reset_index(names="equipment_name"))

# Plot window in local time (counts above and the sums below still use the full query range).
# A full month is too many points for the notebook output limit, so only this window is plotted.
PLOT_START, PLOT_END = "2026-10-07", "2026-10-10"   # inclusive start, exclusive end -> 7th to 9th
win       = (checks["local_time"] >= PLOT_START) & (checks["local_time"] < PLOT_END)
avg_win   = avg_pd[(avg_pd["local_time"] >= PLOT_START) & (avg_pd["local_time"] < PLOT_END)]
final_win = final_pd[(final_pd["local_time"] >= PLOT_START) & (final_pd["local_time"] < PLOT_END)]

# Interactive plot: one panel per stage, only the readings still kept at that stage, coloured by station.
# Drag to zoom (all panels share the time axis), double-click to reset, click a station in the legend to hide/show it.
stations = sorted(checks["equipment_name"].unique())
palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]
colors = {s: palette[i % len(palette)] for i, s in enumerate(stations)}

n_rows = len(stages) + 2
fig = make_subplots(rows=n_rows, cols=1, shared_xaxes=True, vertical_spacing=0.03,
                    subplot_titles=[f"{title} - {int((keep & win).sum())} good readings" for title, keep, _ in stages]
                                   + [f"{avg_title} - {len(avg_win)} values",
                                      f"{final_title} - {int(final_win['final_value'].notna().sum())} values"])
for row, (title, keep, prev) in enumerate(stages, start=1):
    for s in stations:
        g = checks[keep & win & (checks["equipment_name"] == s)]
        fig.add_trace(go.Scattergl(x=g["local_time"], y=g["value"], mode="markers",
                                   marker=dict(size=6 if len(g) < 20 else 3, color=colors[s], opacity=0.7),
                                   name=s, legendgroup=s, showlegend=(row == 1)),
                      row=row, col=1)
    fig.update_yaxes(title_text="W/m²", matches="y", row=row, col=1)

# Panel 4: avg_value, one point per timestamp, coloured by which averaging rule was used
for src in ["measured", "averaged_fallback"]:
    g = avg_win[avg_win["source"] == src]
    fig.add_trace(go.Scattergl(x=g["local_time"], y=g["avg_value"], mode="markers",
                               marker=dict(size=3, color=source_colors[src], opacity=0.7),
                               name=f"avg: {src}", legendgroup=f"avg_{src}"),
                  row=n_rows - 1, col=1)
fig.update_yaxes(title_text="W/m²", matches="y", row=n_rows - 1, col=1)

# Panel 5: final_value per minute, coloured by source (measured / averaged_fallback / interpolated / missing)
for src in ["measured", "averaged_fallback", "interpolated", "missing"]:
    g = final_win[(final_win["source"] == src) & final_win["final_value"].notna()]
    if g.empty:
        continue
    fig.add_trace(go.Scattergl(x=g["local_time"], y=g["final_value"], mode="markers",
                               marker=dict(size=3, color=source_colors[src], opacity=0.7),
                               name=f"final: {src}", legendgroup=f"final_{src}"),
                  row=n_rows, col=1)
fig.update_yaxes(title_text="W/m²", matches="y", row=n_rows, col=1)

fig.update_xaxes(title_text="Local time (America/Chicago)", row=n_rows, col=1)
fig.update_layout(title=f"Hill 1 - {signal} at each cleaning stage, {PLOT_START} to {PLOT_END} (exclusive)", height=300 * n_rows,
                  hovermode="closest", legend=dict(itemsizing="constant"))
fig.show()

# Summation of the final 1-minute values (W/m² per minute). Dividing by 60 gives irradiation in Wh/m².
final_sum = final_pd["final_value"].sum()
print(f"Sum of final_value over {int(final_pd['final_value'].notna().sum())} minutes: {final_sum:,.1f} W/m²·min")
print(f"Total irradiation: {final_sum / 60:,.1f} Wh/m²  ({final_sum / 60 / 1000:,.3f} kWh/m²)")

# Same total per local day
daily = (final_pd.assign(local_date=final_pd["local_time"].dt.date)
                 .groupby("local_date")
                 .agg(minutes=("final_value", "count"), sum_final_value=("final_value", "sum"))
                 .reset_index())
daily["irradiation_kWh_m2"] = daily["sum_final_value"] / 60 / 1000
display(daily)